# 🏆 Gato Mestre / Cartola FC - Pipeline End-to-End de Previsão de Pontuação

Este notebook consolida a execução do **pipeline completo de ponta a ponta** consumindo estritamente a arquitetura modular encapsulada no pacote `src/`.

### 🔄 Fluxo de Execução:
1. **Ingestão Raw**: Carregamento da base histórica bruta (`base_case_gm.csv`) e metadados oficiais da API.
2. **Saneamento e Limpeza Canônica (`src.data_processing`)**: Deduplicação estrita, correção de posições e reconstituição de mando/súmula.
3. **Engenharia de Features (`src.features`)**: Target 2025 homogeneizado, métricas L5, EWMA de piso/teto/disciplina e defasagem temporal.
4. **Modelagem e Validação OOS (`src.models`)**: Split temporal, codificação e treinamento do modelo campeão **LightGBM**.
5. **Exportação Oficial**: Geração e validação do artefato `previsoes.json` no contrato exigido pelo desafio técnico.

## 1. Setup do Ambiente e Importações Modulares de `src/`

In [1]:
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np

# Garante que o diretório raiz do projeto esteja no sys.path
ROOT_DIR = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

# Importação dos módulos oficiais desenvolvidos no pacote src
from src.data_processing.cleaning import limpar_e_preparar_dados
from src.features.engineering import (
    calcular_pontos_target_2025,
    calcular_features_equipe,
    calcular_features_individuais,
    calcular_features_scouts_ewma,
    calcular_features_mercado_e_regime,
    COLS_TO_PREDICT,
    COL_TARGET,
    COLS_IDS,
)
from src.models.model import (
    dividir_dados_temporais,
    preparar_matrizes_arvores,
    treinar_modelo_lightgbm,
    avaliar_modelo,
    gerar_previsoes_json,
)

print(f"✅ Módulos importados com sucesso a partir de: {ROOT_DIR}")

✅ Módulos importados com sucesso a partir de: /Users/actdigital/Documents/desafio-tecnico-gato-mestre-manual


## 2. Ingestão dos Dados Brutos e Metadados Oficiais da API

In [2]:
# Caminhos dos arquivos brutos
RAW_DIR = ROOT_DIR / "data" / "raw"

print("📥 Carregando dataset bruto e metadados canônicos...")
df_raw = pd.read_csv(RAW_DIR / "base_case_gm.csv", low_memory=False)

with open(RAW_DIR / "api_atletas.json", "r", encoding="utf-8") as f:
    api_atletas = json.load(f)

with open(RAW_DIR / "api_jogos.json", "r", encoding="utf-8") as f:
    api_jogos = json.load(f)

with open(RAW_DIR / "api_jogos_detalhes.json", "r", encoding="utf-8") as f:
    api_jogos_detalhes = json.load(f)

print(f"• Registros brutos: {len(df_raw):,}")
print(f"• Atletas na API: {len(api_atletas):,}")
print(f"• Jogos na API: {len(api_jogos):,}")
print(f"• Súmulas detalhadas: {len(api_jogos_detalhes):,}")

📥 Carregando dataset bruto e metadados canônicos...
• Registros brutos: 117,469
• Atletas na API: 391
• Jogos na API: 1,510
• Súmulas detalhadas: 1,510


## 3. Saneamento e Limpeza

In [3]:
print("🧹 Executando pipeline de limpeza canônica...")
df_limpo = limpar_e_preparar_dados(
    df_raw=df_raw,
    api_atletas=api_atletas,
    api_jogos=api_jogos,
    api_jogos_detalhes=api_jogos_detalhes,
)

print("✅ Base limpa consolidada com sucesso!")
print(f"   • Shape: {df_limpo.shape}")
print(f"   • Duplicatas primárias (atleta_id, match_id): {df_limpo.duplicated(subset=['atleta_id', 'match_id']).sum()}")
df_limpo.head(3)

🧹 Executando pipeline de limpeza canônica...
✅ Base limpa consolidada com sucesso!
   • Shape: (115613, 38)
   • Duplicatas primárias (atleta_id, match_id): 0


,atleta_id,apelido,ano,rodada_id,clube_id,posicao_id,status_pre,status_inicial,preco_num,variacao_num,...,PS,GS,GC,CA,CV,FC,I,PP,PC,rodada_cbf_original
0,10001,Elias Ramos,2022,35,115,6,Provável,reserva,5.35,-0.65,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,35
1,10001,Elias Ramos,2022,36,115,6,Provável,reserva,5.08,-0.27,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,36
2,10001,Elias Ramos,2022,37,115,6,Provável,reserva,5.86,0.78,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,37


## 4. Pipeline de Engenharia de Features e Target

In [4]:
print("⚙️ Iniciando Engenharia de Features causal...")
df_features = df_limpo.sort_values(by=["atleta_id", "ano", "rodada_id"]).reset_index(drop=True)
df_features["is_rodada_1"] = (df_features["rodada_id"] == 1).astype(int)

# 1. Target Homogeneizado 2025
df_features = calcular_pontos_target_2025(df_features)

# 2. Força de Equipe e Adversário (L5 com fallback histórico)
df_features = calcular_features_equipe(df_features)

# 3. Features Individuais com Cold Start
df_features = calcular_features_individuais(df_features)

# 4. Decomposição EWMA (Piso, Teto, Chutes, Disciplina, Criação)
df_features = calcular_features_scouts_ewma(df_features)

# 5. Defasagem de Mercado (Lags) e Regimes de Temporada
df_features = calcular_features_mercado_e_regime(df_features)

print(f"✅ Features calculadas com sucesso! Shape final: {df_features.shape}")
print(f"• Total de features preditivas: {len(COLS_TO_PREDICT)}")

⚙️ Iniciando Engenharia de Features causal...
✅ Features calculadas com sucesso! Shape final: (115613, 69)
• Total de features preditivas: 34


## 5. Divisão Temporal e Pré-processamento

In [5]:
# Split estrito temporal: Treino (<= 2023), Validação (2024), Teste OOS (2025)
df_train, df_val, df_test = dividir_dados_temporais(df_features)

# Codificação Categórica sem vazamento de dados
X_train, X_val, X_test, encoder = preparar_matrizes_arvores(
    df_train=df_train,
    df_val=df_val,
    df_test=df_test,
    feature_cols=COLS_TO_PREDICT,
)

y_train = df_train[COL_TARGET].values
y_val = df_val[COL_TARGET].values
y_test = df_test[COL_TARGET].values

print(f"• Matriz Treino: {X_train.shape}")
print(f"• Matriz Validação: {X_val.shape}")
print(f"• Matriz Teste: {X_test.shape}")

📊 Divisão Temporal dos Dados:
   • Treino (<= 2023): 59,213 registros
   • Validação (2024): 28,568 registros
   • Teste Out-of-Time (2025): 27,832 registros
• Matriz Treino: (59213, 34)
• Matriz Validação: (28568, 34)
• Matriz Teste: (27832, 34)


## 6. Treinamento do Modelo Campeão (LightGBM) e Avaliação Out-of-Time

In [8]:
# Treinamento do LightGBM com hiperparâmetros oficiais
modelo_lgbm = treinar_modelo_lightgbm(
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
)

# Inferência no conjunto de Teste OOS (2025)
y_pred_test = modelo_lgbm.predict(X_test)

# Avaliação com as métricas de negócio e erro residual
metricas_teste = avaliar_modelo(y_test, y_pred_test, nome_conjunto="Teste OOS 2025")


🚀 Treinando LightGBM Regressor...


/Users/actdigital/Documents/desafio-tecnico-gato-mestre-manual/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


✅ Treinamento LightGBM finalizado com sucesso!

📈 Métricas de Avaliação - Teste OOS 2025:
   • MAE         : 1.4541
   • RMSE        : 2.4704
   • R2          : 0.3388
   • Pearson_r   : 0.5842
   • MAE_Top20%  : 3.2941


## 7. Geração e Validação do Contrato Oficial `previsoes.json`

In [ ]:
CAMINHO_PREVISOES = ROOT_DIR / "previsoes.json"

# Geração do arquivo JSON no dir do desafio
resultado_json = gerar_previsoes_json(
    df_test=df_test,
    y_pred=y_pred_test,
    output_path=str(CAMINHO_PREVISOES),
)

# Validação de conformidade e integridade
assert CAMINHO_PREVISOES.exists(), "Erro: previsoes.json não foi criado."
assert "previsoes" in resultado_json, "Erro: chave previsoes ausente no JSON."
assert len(resultado_json["previsoes"]) == len(df_test), "Erro: contagem de previsões diverge do conjunto de teste."

amostra = resultado_json["previsoes"][0]
campos_esperados = {"atleta_id", "ano", "rodada_id", "clube_id", "posicao_id", "pontos_predito", "data_predicao"}
assert campos_esperados.issubset(amostra.keys()), f"Campos ausentes na previsão: {campos_esperados - set(amostra.keys())}"

print("\n🎉 PIPELINE FINALIZADA COM 100% DE CONFORMIDADE!")
print(f"• Arquivo gerado: {CAMINHO_PREVISOES}")
print(f"• Total de atletas/rodadas preditos: {len(resultado_json['previsoes']):,}")
print(f"• Exemplo do primeiro registro: {amostra}")


💾 Arquivo de previsões salvo em: /Users/actdigital/Documents/desafio-tecnico-gato-mestre-manual/previsoes.json
   • Total de previsões geradas: 27,832

🎉 PIPELINE FINALIZADA COM 100% DE CONFORMIDADE!
• Arquivo gerado: /Users/actdigital/Documents/desafio-tecnico-gato-mestre-manual/previsoes.json
• Total de atletas/rodadas preditos: 27,832
• Exemplo do primeiro registro: {'atleta_id': 10004, 'ano': 2025, 'rodada_id': 1, 'clube_id': 105, 'posicao_id': 6, 'pontos_predito': 2.95, 'data_predicao': '2026-08-20T00:58:29Z'}
